In [31]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

# ==========================================
# 1. ROBUST DATA PREPARATION FUNCTIONS
# ==========================================

def load_and_preprocess_raw_data(filepath, sample_size=100000):
    # Load raw data with pipes as separator
    df = pd.read_csv(filepath, sep="|")
    
    # Drop rows where target values are missing
    df = df.dropna(subset=["TotalPremium", "TotalClaims"])
    
    # Downsample safely to prevent memory errors
    if sample_size and len(df) > sample_size:
        df = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
    
    # Impute numeric columns with median
    num_cols = df.select_dtypes(include=np.number).columns
    for col in num_cols:
        df[col] = df[col].fillna(df[col].median())
        
    # Impute categorical columns with mode
    cat_cols = df.select_dtypes(include="object").columns
    for col in cat_cols:
        df[col] = df[col].fillna(df[col].mode()[0])
        
    # Feature Engineering
    if "RegistrationYear" in df.columns:
        df["vehicle_age"] = 2015 - df["RegistrationYear"]
        df["vehicle_age"] = df["vehicle_age"].clip(lower=0)
    else:
        df["vehicle_age"] = 0
        
    df["has_claim"] = (df["TotalClaims"] > 0).astype(int)
    
    return df

def prepare_features_and_targets(df):
    # Define targets and metrics to drop from the training features X
    leakage_cols = ["TotalPremium", "TotalClaims", "has_claim", "margin", "loss_ratio"]
    
    X = df.drop(columns=[col for col in leakage_cols if col in df.columns], errors="ignore")
    y_class = df["has_claim"]
    y_reg = df["TotalClaims"]
    
    # Identify categorical columns
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    
    # Separate high and low cardinality columns
    high_card_cols = []
    low_card_cols = []
    for col in cat_cols:
        if X[col].nunique() > 15:
            high_card_cols.append(col)
        else:
            low_card_cols.append(col)
            
    # Apply Frequency Encoding for high cardinality features
    for col in high_card_cols:
        freq = X[col].value_counts(normalize=True)
        X[col + "_freq"] = X[col].map(freq)
        X = X.drop(columns=[col])
        
    # Apply One-Hot Encoding for low cardinality features
    X = pd.get_dummies(X, columns=low_card_cols, drop_first=True)
    
    # Convert all boolean columns to integers (0 or 1)
    bool_cols = X.select_dtypes(include=["bool"]).columns
    X[bool_cols] = X[bool_cols].astype(int)
    
    # ----------------------------------------------------------------
    # DIAGNOSTIC & ROBUST REPLACEMENT FOR INF AND NAN VALUES
    # ----------------------------------------------------------------
    # 1. Replace any infinite values with NaN
    X = X.replace([np.inf, -np.inf], np.nan)
    
    # 2. Identify which columns contain NaNs before cleaning
    nan_columns_before = X.columns[X.isnull().any()].tolist()
    if len(nan_columns_before) > 0:
        print(f"[Diagnostic] Found NaNs in the following columns: {nan_columns_before}")
    
    # 3. Fill NaNs with column medians
    X = X.fillna(X.median())
    
    # 4. Fallback fill with 0 (for any column where median was NaN or undefined)
    X = X.fillna(0)
    
    # 5. Final validation check
    nan_columns_after = X.columns[X.isnull().any()].tolist()
    if len(nan_columns_after) > 0:
        print(f"[Warning] Failed to clean NaNs in columns: {nan_columns_after}")
    else:
        print("[Success] All NaNs and Infs successfully cleared from the feature matrix.")
        
    return X, y_class, y_reg

# ==========================================
# 2. RUN CLASSIFICATION PIPELINE
# ==========================================

# Preprocess and prepare data
df_clean = load_and_preprocess_raw_data("../data/MachineLearningRating_v3.txt", sample_size=100000)
X, y_class, y_reg = prepare_features_and_targets(df_clean)

# Split the dataset
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

# Define Classifiers
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
}

# Train and evaluate
classification_results = []
trained_classifiers = {}

for name, model in classifiers.items():
    model.fit(X_train_c, y_train_c)
    probs = model.predict_proba(X_test_c)[:, 1]
    preds = model.predict(X_test_c)
    
    auc = roc_auc_score(y_test_c, probs)
    f1 = f1_score(y_test_c, preds, zero_division=0)
    prec = precision_score(y_test_c, preds, zero_division=0)
    rec = recall_score(y_test_c, preds, zero_division=0)
    
    trained_classifiers[name] = model
    classification_results.append({
        "Model": name, 
        "ROC-AUC": auc, 
        "F1-Score": f1,
        "Precision": prec,
        "Recall": rec
    })

df_class_results = pd.DataFrame(classification_results)
print("\n--- Classification Results (Claim Probability) ---")
print(df_class_results)

C:\Users\YOGA 9I\AppData\Local\Temp\ipykernel_20940\3859372007.py:15: DtypeWarning: Columns (0: CapitalOutstanding, 1: CrossBorder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath, sep="|")
C:\Users\YOGA 9I\AppData\Local\Temp\ipykernel_20940\3859372007.py:30: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns
C:\Users\YOGA 9I\AppData\Local\Temp\ipykernel_20940\3859372007.py:54: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'ob

[Diagnostic] Found NaNs in the following columns: ['NumberOfVehiclesInFleet']
[Success] All NaNs and Infs successfully cleared from the feature matrix.

--- Classification Results (Claim Probability) ---
                 Model   ROC-AUC  F1-Score  Precision  Recall
0  Logistic Regression  0.849580       0.0        0.0     0.0
1        Random Forest  0.895695       0.0        0.0     0.0
2              XGBoost  0.905006       0.0        0.0     0.0


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

# ==========================================
# 1. ROBUST DATA PREPARATION FUNCTIONS
# ==========================================

def load_and_preprocess_raw_data(filepath, sample_size=100000):
    # Load raw data with pipes as separator
    df = pd.read_csv(filepath, sep="|")
    
    # Drop rows where target values are missing
    df = df.dropna(subset=["TotalPremium", "TotalClaims"])
    
    # Downsample safely to prevent memory errors
    if sample_size and len(df) > sample_size:
        df = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
    
    # Impute numeric columns with median
    num_cols = df.select_dtypes(include=np.number).columns
    for col in num_cols:
        df[col] = df[col].fillna(df[col].median())
        
    # Impute categorical columns with mode
    cat_cols = df.select_dtypes(include="object").columns
    for col in cat_cols:
        df[col] = df[col].fillna(df[col].mode()[0])
        
    # Feature Engineering
    if "RegistrationYear" in df.columns:
        df["vehicle_age"] = 2015 - df["RegistrationYear"]
        df["vehicle_age"] = df["vehicle_age"].clip(lower=0)
    else:
        df["vehicle_age"] = 0
        
    df["has_claim"] = (df["TotalClaims"] > 0).astype(int)
    
    return df

def prepare_features_and_targets(df):
    # Define targets and metrics to drop from the training features X
    leakage_cols = ["TotalPremium", "TotalClaims", "has_claim", "margin", "loss_ratio"]
    
    X = df.drop(columns=[col for col in leakage_cols if col in df.columns], errors="ignore")
    y_class = df["has_claim"]
    y_reg = df["TotalClaims"]
    
    # Identify categorical columns
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    
    # Separate high and low cardinality columns
    high_card_cols = []
    low_card_cols = []
    for col in cat_cols:
        if X[col].nunique() > 15:
            high_card_cols.append(col)
        else:
            low_card_cols.append(col)
            
    # Apply Frequency Encoding for high cardinality features
    for col in high_card_cols:
        freq = X[col].value_counts(normalize=True)
        X[col + "_freq"] = X[col].map(freq)
        X = X.drop(columns=[col])
        
    # Apply One-Hot Encoding for low cardinality features
    X = pd.get_dummies(X, columns=low_card_cols, drop_first=True)
    
    # Convert all boolean columns to integers (0 or 1)
    bool_cols = X.select_dtypes(include=["bool"]).columns
    X[bool_cols] = X[bool_cols].astype(int)
    
    # Replace any infinite values with NaN
    X = X.replace([np.inf, -np.inf], np.nan)
    
    # Identify which columns contain NaNs before cleaning
    nan_columns_before = X.columns[X.isnull().any()].tolist()
    if len(nan_columns_before) > 0:
        print(f"[Diagnostic] Found NaNs in the following columns before final imputation: {nan_columns_before}")
    
    # Fill NaNs with column medians
    X = X.fillna(X.median())
    
    # Fallback fill with 0 (for any column where median was NaN or undefined)
    X = X.fillna(0)
    
    # Final validation check
    nan_columns_after = X.columns[X.isnull().any()].tolist()
    if len(nan_columns_after) > 0:
        print(f"[Warning] Failed to clean NaNs in columns: {nan_columns_after}")
    else:
        print("[Success] All NaNs and Infs successfully cleared from the feature matrix.")
        
    return X, y_class, y_reg

# ==========================================
# 2. RUN CLASSIFICATION PIPELINE
# ==========================================

print("Starting pipeline from scratch...")

# Load and preprocess raw data
df_clean = load_and_preprocess_raw_data("../data/MachineLearningRating_v3.txt", sample_size=100000)

# Process features and targets (this overwrites the X, y_class, and y_reg variables in memory)
X, y_class, y_reg = prepare_features_and_targets(df_clean)

# Split the dataset
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

# Define Classifiers
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
}

# Train and evaluate
classification_results = []
trained_classifiers = {}

for name, model in classifiers.items():
    print(f"Training {name}...")
    model.fit(X_train_c, y_train_c)
    probs = model.predict_proba(X_test_c)[:, 1]
    preds = model.predict(X_test_c)
    
    auc = roc_auc_score(y_test_c, probs)
    f1 = f1_score(y_test_c, preds, zero_division=0)
    prec = precision_score(y_test_c, preds, zero_division=0)
    rec = recall_score(y_test_c, preds, zero_division=0)
    
    trained_classifiers[name] = model
    classification_results.append({
        "Model": name, 
        "ROC-AUC": auc, 
        "F1-Score": f1,
        "Precision": prec,
        "Recall": rec
    })

df_class_results = pd.DataFrame(classification_results)
print("\n--- Classification Results (Claim Probability) ---")
print(df_class_results)

Starting pipeline from scratch...


C:\Users\YOGA 9I\AppData\Local\Temp\ipykernel_9064\167790242.py:15: DtypeWarning: Columns (0: CapitalOutstanding, 1: CrossBorder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath, sep="|")
C:\Users\YOGA 9I\AppData\Local\Temp\ipykernel_9064\167790242.py:30: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns
C:\Users\YOGA 9I\AppData\Local\Temp\ipykernel_9064\167790242.py:54: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' 

[Diagnostic] Found NaNs in the following columns before final imputation: ['NumberOfVehiclesInFleet']
[Success] All NaNs and Infs successfully cleared from the feature matrix.
Training Logistic Regression...
Training Random Forest...
Training XGBoost...

--- Classification Results (Claim Probability) ---
                 Model   ROC-AUC  F1-Score  Precision  Recall
0  Logistic Regression  0.849580       0.0        0.0     0.0
1        Random Forest  0.895695       0.0        0.0     0.0
2              XGBoost  0.905006       0.0        0.0     0.0


In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

# ==========================================
# 1. TRAIN CLAIM SEVERITY MODELS (REGRESSION)
# ==========================================
print("Starting severity regression training...")

# Filter only rows where a claim actually occurred (TotalClaims > 0)
claim_mask = (y_reg > 0)
X_reg = X[claim_mask]
y_reg_sub = y_reg[claim_mask]

print(f"Total number of policies with claims for regression: {len(X_reg)}")

# Split severity data (80:20)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg_sub, test_size=0.2, random_state=42
)

# Define Regressors
regressors = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
}

regression_results = []
trained_regressors = {}

for name, model in regressors.items():
    print(f"Training {name}...")
    model.fit(X_train_r, y_train_r)
    preds = model.predict(X_test_r)
    
    # Clip predictions to 0 as claims cannot be negative values
    preds = np.clip(preds, 0, None)
    
    rmse = np.sqrt(mean_squared_error(y_test_r, preds))
    r2 = r2_score(y_test_r, preds)
    
    trained_regressors[name] = model
    regression_results.append({
        "Model": name,
        "RMSE": rmse,
        "R2": r2
    })

df_reg_results = pd.DataFrame(regression_results)
print("\n--- Regression Results (Claim Severity where Claim > 0) ---")
print(df_reg_results)


# ==========================================
# 2. PREMIUM OPTIMIZATION FRAMEWORK
# ==========================================
print("\nCalculating optimized premiums...")

# Select the best performing models (typically XGBoost for both)
best_cls_model = trained_classifiers["XGBoost"]
best_reg_model = trained_regressors["XGBoost"]

# 1. Predict probability of claim on classification test set
p_claim = best_cls_model.predict_proba(X_test_c)[:, 1]

# 2. Predict expected severity on classification test set
predicted_severity = np.clip(best_reg_model.predict(X_test_c), 0, None)

# 3. Calculate Pure Premium (Expected Loss)
pure_premium = p_claim * predicted_severity

# 4. Add Expense Loading (e.g., 20% of pure premium) and Profit Margin (e.g., 10%)
expense_loading = 0.20 * pure_premium
profit_margin = 0.10 * pure_premium

# 5. Calculate Final Recommended Premium
calculated_premium = pure_premium + expense_loading + profit_margin

# Create comparative dataframe using original index values
pricing_comparison = pd.DataFrame({
    "Original_Premium": df_clean.loc[X_test_c.index, "TotalPremium"],
    "Actual_Claims": df_clean.loc[X_test_c.index, "TotalClaims"],
    "Predicted_Prob_Claim": p_claim,
    "Predicted_Severity": predicted_severity,
    "Calculated_Premium": calculated_premium
}).reset_index(drop=True)

print("\n--- Pricing Comparison (Sample Output) ---")
print(pricing_comparison.head(15))

Starting severity regression training...
Total number of policies with claims for regression: 294
Training Linear Regression...
Training Random Forest...
Training XGBoost...

--- Regression Results (Claim Severity where Claim > 0) ---
               Model          RMSE        R2
0  Linear Regression  24666.265589 -0.029299
1      Random Forest  22244.902040  0.162864
2            XGBoost  20201.979981  0.309565

Calculating optimized premiums...

--- Pricing Comparison (Sample Output) ---
    Original_Premium  Actual_Claims  Predicted_Prob_Claim  Predicted_Severity  \
0           5.133596            0.0              0.000045         2958.051758   
1           7.053070            0.0              0.000062         5398.264648   
2          21.929825            0.0              0.013330         2006.182007   
3           0.000000            0.0              0.001322        23169.947266   
4           0.000000            0.0              0.000057         4435.100586   
5           2.437456